# Exploring the Point Generators

In this notebook we explore the three different point generator classes.

1. The *PointGenerator* class is the base class the other two inherit from. It can be used for Complete Intersection Calabi-Yau (CICY) manifolds with a single hypersurface, e.g. the quintic, bicubic, p1p3, the tetraquadric, or a torus, some K3s ...
2. There is the *CICYPointGenerator* which works for any CICY.
3. There is the *PointGeneratorMathematica* which works for any CICY (also for just a single hypersurface. It requires a Mathematica installation which will be used as a backend for the computations.
4. There is the *ToricPointGenerator* which can generate points on any CY coming from the Kreuzer-Skarke list. NOTE: as of v0.0.1 there are still some issues with finding the correct integration weights from generalizing a theorem by Shiffman and Zelditch. For the correct implementation use *ToricPointGeneratorMathematica*.

The Fermat quintic manifold can be implemented in each of the three PointGenerator classes and will be used as an example.

Recall that the Fermat quintic is given by the following polynomial:

$$
Q = \sum_i^5 z_i^5
$$

in a $\mathbb{P}^4$ ambient space.

In [2]:
import numpy as np
import os
import pickle

## PointGenerator

First we load the PointGenerator from the cymetric package.

In [2]:
from cymetric.pointgen.pointgen import PointGenerator

In general all routines and classes of the cymetric package will have a brief description of their functionality and their arguments. You can access it with the help function.

In [ ]:
help(PointGenerator)

In the next step we define the five defining monomials, the coefficients in front of each monomial, the single Kähler moduli, and the ambient space projective factor(s):

In [3]:
monomials = 5 * np.eye(5, dtype=np.int64)
coefficients = np.ones(5)
kmoduli = np.ones(1)
ambient = np.array([4])

We can now initiate the PointGenerator

In [5]:
pg = PointGenerator(monomials, coefficients, kmoduli, ambient)

and generate some (100) points:

In [6]:
points = pg.generate_points(100)
points[:10]

Generating points: 105pt [00:04, 21.77pt/s]                       


array([[-0.26571881-9.17297455e-01j, -0.36345359+2.42049070e-01j,
         1.        -1.92084502e-17j, -0.59842131-1.43934436e-01j,
        -0.35605605-6.91997143e-01j],
       [ 1.        -5.14815752e-17j, -0.31804611-5.70832600e-01j,
         0.78152219+6.00384439e-01j,  0.15162194-1.69715014e-02j,
         0.31991803-3.70442042e-01j],
       [ 1.        +1.94547509e-17j, -0.23516908-2.60266458e-01j,
        -0.77837294+1.05176966e-01j,  0.81948808-4.76754593e-01j,
         0.66788379+2.93563540e-01j],
       [ 0.8244501 +5.64605799e-01j, -0.16831985-6.66244130e-01j,
         1.        -1.18234507e-17j,  0.28974831+6.27956749e-01j,
        -0.05495057-4.40566702e-02j],
       [-0.18569486+7.50333720e-01j,  0.24012511-3.46593059e-01j,
         1.        +1.30509327e-17j, -0.31459026+8.85538129e-01j,
        -0.45377699-5.09986505e-02j],
       [-0.62309363+5.81626089e-02j, -0.4679331 +6.79700462e-02j,
        -0.54795213-2.22345813e-02j,  1.        -8.92823695e-19j,
        -0.9677924

We see that the largest coordinate of each point is 1+0.j. The reason for that is two-fold:

1. The numerics are more stable when we work with points, which have affine coordinates in the range of $|x_i| < 1$.
2. The coordinate with 1+0.j already specifies the ambient space patch we are working in.

We can also check if these points are all really satisfying the hypersurface equation of the Calabi-Yau.

In [7]:
np.allclose(np.abs(pg.cy_condition(points)), np.zeros_like(pg.cy_condition(points)))

True

What we are really interested in from the *PointGenerator* is a training set for our neural networks. Such a training set can be generated as follows:

In [8]:
help(pg.prepare_dataset)

Help on method prepare_dataset in module cymetric.pointgen.pointgen:

prepare_dataset(n_p, dirname, val_split=0.1, ltails=0, rtails=0) method of cymetric.pointgen.pointgen.PointGenerator instance
    Prepares training and validation data.
    
    Args:
        n_p (int): Number of points to generate.
        dirname (str): Directory name to save dataset in.
        val_split (float, optional): train-val split. Defaults to 0.1.
        ltails (float, optional): Percentage discarded on the left tail
            of weight distribution. Defaults to 0.
        rtails (float, optional): Percentage discarded on the right tail
            of weight distribution. Defaults to 0.
    
    Returns:
        np.float: kappa = vol_k / vol_cy



We specify the number of points and the directory name to save the file in. Note the file will always have the name *dataset.npz*.

In [4]:
dirname = 'fermat_pg'
n_p = 100000

and generate the dataset. This will also compute and return $\kappa=\text{vol}_\text{K}/\text{vol}_\text{CY}$

In [10]:
kappa = pg.prepare_dataset(n_p, dirname)
kappa

Generating points: 100005pt [00:00, 247652.56pt/s]                           
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen.py:1071: ComplexWarning: Casting complex values to real discards the imaginary part
  point_weights['point'], point_weights['weight'] = points[0:n_p], weights[0:n_p]
pointgen:INFO:Vol_k: 5.000000000000001, Vol_cy: 284.2561104432774.


0.017589771393842168

We load the dataset with

In [11]:
data = np.load(os.path.join(dirname, 'dataset.npz'))

and study its content

In [12]:
for key in data:
    print(key)

X_train
y_train
X_val
y_val
val_pullbacks


It contains training and validation data and the validation pullbacks. You might ask, what is written in the y_true values for validation and training data, given that we don't know the exact Ricci-flat metric. The 'y_train/val' arrays contain the integration weights and $\Omega \wedge \bar\Omega$ for each point. In principle, they can be used for any relevant pointwise information that could be needed during the training process.

In [13]:
weights = data['y_val'][:, 0]
omega = data['y_val'][:, 1]

we can also compute these values directly with the *PointGenerator*. Note, that the points in 'X_train/val' are floats, because our neural network will work with real values. We can recover the complex points as follows:

In [14]:
points = data['X_val'][:, 0:pg.ncoords] + 1.j * data['X_val'][:, pg.ncoords:]

and then compute the weights

In [15]:
weights2 = pg.point_weight(points)
np.allclose(weights, weights2)

True

and the holomorphic volume form

In [16]:
omega2 = pg.holomorphic_volume_form(points)
omega2 = omega2 * np.conj(omega2)
np.allclose(omega, omega2)

True

We will have to give information of the monomials and their derivatives to the tensorflow model. For this purpose we will create pickled dictionary denoted by *BASIS*.

In [17]:
pg.prepare_basis(dirname, kappa=kappa)

0

Let us look at the information stored in *basis.pickle*

In [18]:
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA


So in case you want to use your own PointGenerator with our TensorFlow models you will have to create a similar basis dictionary. Here we briefly describe what each of these keys stands for. In general *Q* denotes the defining hypersurface(s) with the final integer digit denoting the hypersurface index. *D* refers to derivatives, *Z* to the ambient space coordinates, *B* to a monomials basis and *F* for factors/coefficients for each monomial.

1. "DQDZB0": $\frac{\partial Q_0}{\partial z_i}$ monomials basis of derivatives of the first (and only for quintic) hypersurface w.r.t. ambient coordinates. 
2. "DQDZF0": $\frac{\partial Q_0}{\partial z_i}$ coefficients of the monomial basis.
3. "QB0": monomials basis for first hypersurface
4. "QF0": coefficients for monomial basis.
5. "NFOLD": CY dimension.
6. "AMBIENT": degrees of projective spaces making up the ambient space.
7. "KMODULI": kähler moduli corresponding to each projective factors. Note the CY needs to be favourable, otherwise you will have some superposition.
8. "NHYPER": number of hypersurfaces.
9. "INTNUMS": The intersection numbers of the CY.
10. "KAPPA": The ratio between the volume measures (Kahler volume over holomorphic volume)

That pretty much sums up our introduction to the *PointGenerator* class, next we will implement the Fermat quintic in the *CICYPointGenerator*.

## CICYPointGenerator

The other point generators come with the same functionality and routines as the *PointGenerator*. 

First we load from the cymetric package.

In [19]:
from cymetric.pointgen.pointgen_cicy import CICYPointGenerator

In contrast to the *PointGenerator* the *CICYPointGenerator* expects a list of monomials and coefficients. We reuse our previous monomials (and coefficients) with

In [20]:
pgcicy = CICYPointGenerator([monomials], [coefficients], kmoduli, ambient)

we again create a dataset

In [21]:
dirname = 'fermat_pgcicy'

In [22]:
kappa = pgcicy.prepare_dataset(n_p, dirname)

Generating points:   0%|          | 0/1 [00:00<?, ?assignment/s]/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen_cicy.py:417: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = opt.fsolve(
/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen_cicy.py:417: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = opt.fsolve(
/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen_cicy.py:417: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = opt.fsolve(
/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen_cicy.py:417: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  sol = opt.fsolve(
/Users/ruehle/GitHub/cymetric/cymetric/pointgen/pointgen_cicy.py:417: Run

As you might have realised, this took significantly longer than before. The reason for that is also related to all the warnings. The *CICYPointGenerator* utilizes *scipy.optimize.fsolve* to find solutions on CICYs. *fsolve* only provides a single root of the defining hypersurfaces and requires more involved numerics, thus leading to worse performance (also in accuracy).

We again create a basis for the tensorflow models

In [23]:
pgcicy.prepare_basis(dirname, kappa)
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA


and see that the keys are identical to before.

## PointGeneratorMathematica

As the name suggests, *PointGeneratorMathematica* uses Mathematica as a backend and hence needs mathematica installed on the system. It uses [wolframclient](https://reference.wolfram.com/language/WolframClientForPython/) for interop.

In [5]:
from cymetric.pointgen.pointgen_mathematica import PointGeneratorMathematica

Just like the CICYPointGenerator, *PointGeneratorMathematica* expects a list of monomials and coefficients. Depending on the setup of the machine, one needs to specify the path to the Mathematica kernel. We reuse our previous monomials (and coefficients) with

In [ ]:
kernel_path = None
# kernel_path = "/Applications/Wolfram.app/Contents/MacOS/WolframKernel"  # default for MacOS, adjust if needed or try None  
pgmath = PointGeneratorMathematica([monomials], [coefficients], kmoduli, ambient, kernel_path=kernel_path)

we again create a dataset

In [7]:
dirname = 'fermat_pgmath'

In [8]:
kappa = pgmath.prepare_dataset(n_p, dirname)

WolframKernel-<tcp://127.0.0.1:52870>:DEBUG:Evaluating a new expression.
pointgenMathematica:DEBUG:Running with 16 Mathematica kernels.
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Configuration matrix: {{5}}
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Number of Parameters per P^n: {1}
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Number of points on CY from one ambient space intersection: 5
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Now generating 100000 points...
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 0% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 5% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 10% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 15% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 20% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 25% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 30% of points
WolframKernel-<tcp://127.0.0.1:52870>:INFO:Generated 35% of points
Wo

We again create a basis for the tensorflow models

In [9]:
pgmath.prepare_basis(dirname, kappa)
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA


## CICYPointGeneratorMC

The CICYPointGeneratorMC works like the CICY point generator, but it uses Monte Carlo sampling instead of the Shiffman-Zelditch theorem to sample points. For CICYs, this is often slower, but it gives a point sample that is distributed with respect to the CY metric, instead of the FS metric. Hence, all weights are one (or the same constant after normalization)

In [11]:
from cymetric.pointgen.pointgen_mc import CICYPointGeneratorMC

The arguments for CICYPointGeneratorMC are the same as for CICYPointGenerator, you can also manually specify the burn-in and the thinning of the sampler. If you do not specify these by hand, the algorithm will detect reasonable values automatically.

In [13]:
pgmc = CICYPointGeneratorMC([monomials], [coefficients], kmoduli, ambient)
dirname = 'fermat_pgmc'
kappa = pgmc.prepare_dataset(n_p, dirname)

MCpointgen:INFO:Finding initial point...
MCpointgen:INFO:Tuning step size (target acceptance 30%)...
MCpointgen:INFO:  -> step_size = 31.68
MCpointgen:INFO:Estimating thinning from autocorrelation...
MCpointgen:INFO:  -> thin = 2
MCpointgen:INFO:Running chain: burn_in = 100, thin = 2, n_p = 100000
MCMC (sampling): 100%|██████████| 200100/200100 [02:03<00:00, 1623.77step/s, accept=80.6%, pts=1e+5] 
MCpointgen:INFO:Acceptance rate: 0.806 (161351/200100)
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [14]:
pgmc.prepare_basis(dirname, kappa)
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA


## ToricPointGeneratorMathematica

The *ToricPointGeneratorMathematica* requires both [SageMath](https://www.sagemath.org/) and Mathematica. SageMath does the toric computations and Mathematica the rest. The *toric_data* can be straightforwardly generated in any sage kernel that has access to the *cymetric* package. In praxis only a single module is needed which can be found [here](../cymetric/sage/sagelib.py). Alternatively, you can run the generator directly from within SageMath, as illustrated in another notebook.

The next cell won't work in your regular python notebook because it requires some sage routines for toric geometry and Mathematica. See [here](https://doc.sagemath.org/html/en/reference/schemes/sage/schemes/toric/variety.html) for more information about toric varieties and their implementation in sage and [here](https://doc.sagemath.org/html/en/reference/discrete_geometry/sage/geometry/triangulation/point_configuration.html) for information about triangulations of PointCollections. For Mathematica interop, it uses the [wolframclient](https://reference.wolfram.com/language/WolframClientForPython/) for interop.

We begin by setting up the quintic vertices, which define the fan of the toric ambient variety. After initialising said variety we load the *prepare_toric_cy_data()* routine and generate the neccessary data for the *ToricPointGenerator* and the toric TensorFlow models.

In [ ]:
try:
    from cymetric.sage.sagelib import prepare_toric_cy_data
    import os
    # Quintic vertices
    vertices = [
    [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1], [-1, -1, -1, -1]
    ]
    polytope = LatticePolytope(vertices)
    all_pts = set(map(tuple, polytope.points()))

    # Discard points strictly interior to codim-1 faces (= facets)
    codim1_interior = set()
    for facet in polytope.facets():
        for p in facet.interior_points():
            codim1_interior.add(tuple(p))

    # Keep everything else
    cy_points = [p for p in polytope.points() if tuple(p) not in codim1_interior]
    pConfig = PointConfiguration(cy_points, star=[0 for _ in range(len(cy_points[0]))])
    try:  # fails if TOPCOM is not installed. In that case, we omit the (actually important) regularity check
        triangs = pConfig.restrict_to_connected_triangulations().restrict_to_fine_triangulations().restrict_to_regular_triangulations().restrict_to_star_triangulations([0 for _ in range(len(vertices[0]))]).triangulations_list()
    except:
        triangs = pConfig.restrict_to_connected_triangulations().restrict_to_fine_triangulations().restrict_to_star_triangulations([0 for _ in range(len(vertices[0]))]).triangulations_list()
    triang = triangs[0]
    tv_fan = triang.fan()
    tv = ToricVariety(tv_fan)

    fname = os.path.join('fermat_pgtoricmath', 'toric_data.pickle')
    toric_data = prepare_toric_cy_data(tv, fname)
except:
    print("Sage not available, execute this within a Sage kernel.")

We are now in a position to go back to our regular python kernel or simply continue in Sage and load the *PointGeneratorToricMathematica*.

In [2]:
from cymetric.pointgen.pointgen_mathematica import ToricPointGeneratorMathematica
import numpy as np
import os
import pickle

Let us look at the information that has been written to *toric_data.pickle*.

In [3]:
dirname = 'fermat_pgtoricmath'
with open(os.path.join(dirname, 'toric_data.pickle'), 'rb') as f:
    toric_data = pickle.load(f)
for key in toric_data:
    print(key)
    print(toric_data[key])

dim_cy
3
vol_j_norm
5
coeff_aK
[(-1.2107783499978841+3.0755417042654605j), (-1.7461964893542359-0.2958439132080485j), (-0.017382319548046695-1.5885147385127858j), (-1.1056651163530131-0.8594434006388215j), (-1.0773270752014996-0.44963898114534623j), (0.6342862295390801-0.6810665923093405j), (-1.463937730095888+0.877697319627733j), (-0.7122473741748493+1.759470378596264j), (0.23108940772170963+0.5912131528277726j), (-0.6908252698293487-0.6715487057800472j), (1.0262293827358568-1.9165359557710742j), (-0.6144349928998802+1.1315345315388472j), (0.8801973994604347+2.470317194746462j), (0.21613556018435437-1.3073991244171983j), (1.4581254526373584+0.5826760771808603j), (-0.35895813973212387-0.9595222194481265j), (-1.1448813789597416-0.0665174122260354j), (0.6867618147009131+0.7425453871359471j), (0.6752758348270502-0.33171461955608494j), (0.05000922129979441-0.7173088466165982j), (-0.12367348745995488+0.647799645145801j), (0.6686491786324991+0.712628149227071j), (-0.5977622437209195-1.078451

We have the following keys:

1. "dim_cy": contains the dimension of the Calabi-Yau.
2. "vol_j_norm": information for normalization of weights.
2. "coeff_aK": are generic complex coefficients in front of each of the defining hypersurface monomials, which you get from the Batyrev construction. Note: Those are by default complex valued as they represent a (redundant) description of the complex moduli.
3. "exp_aK": is the monomial basis for the defining equation.
4. "exps_sections": is a monomial basis for the sections of the kähler cone generators. This one will be needed to generate the integration weights and a Kähler metric in the same Kähler class as our Ricci-flat metric.
5. "patch_masks": are (boolean) coordinate masks for all the patches in the TV.
6. "glsm_charges": are the GLSM charges of the TV.
7. "non_ci_coeffs": when sampling points, we use the projecitivized 0th cohomology class of the Kahler cone generators. There can be non-complete-intersection relations among the sections, which are encoded in the non_ci_coeffs and the non_ci_exps.
8. "non_ci_exps": The exponents of all Kahler cone generator sections
9. "int_nums": are the intersection numbers of the TV.

Having loaded the toric_data we can initialize the ToricPointGenerator

In [ ]:
coefficients = np.ones(5)
monomials = 5 * np.eye(5, dtype=np.int64)
kmoduli = np.ones(1)

kernel_path = None
# kernel_path = "/Applications/Wolfram.app/Contents/MacOS/WolframKernel"  # default for MacOS, adjust if needed or try None  
pgtoricmath = ToricPointGeneratorMathematica(toric_data, kmoduli, kernel_path=kernel_path)

We now have a generic quintic with coefficients in front of all 125 monomials.

What if we only want the Fermat Quintic? We can override the information in "coeff_aK" and "exp_aK" with e.g.

In [7]:
toric_data["coeff_aK"] = coefficients
toric_data["exp_aK"] = monomials
n_p = 100000

we continue just as before with creating a dataset

In [8]:
kappa_toric = pgtoricmath.prepare_dataset(n_p, dirname)

WolframKernel-<tcp://127.0.0.1:53143>:INFO:Connected to logging socket: tcp://127.0.0.1:53143
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Number of points on CY from one ambient space intersection: 5
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Now generating 100000 points...
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 0% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 5% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 10% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 15% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 20% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 25% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 30% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 35% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 40% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 45% of points
WolframKernel-<tcp://127.0.0.1:53143>:INFO:Generated 50% o

We note, that the data generation again took quite some time even though there should just be a single hypersurface. We also optimize w.r.t. to the sections coming from the Kähler generators and thus have more than a single hypersurface to consider.

Also, if you paid close attention to the output, you might have noticed that both Vol_k and Vol_cy are larger by a factor of 5 as compared to the other point generators. This is because the toric model does calculate the correct normalization (i.e. such that vol(X)$=\int J^3=d_{ijk}t^ict^j t^k=\text{mean}(\text{det}(g)/|\Omega|^2 w$) from the triple intersection numbers. To get the same behavior for the other point generators, you need to pass the argument vol_j_norm=5 to the generators and then call prepare_dataset with normalize_to_vol_j=True.

We create the *BASIS* containing all information for the TensorFlow models:

In [9]:
pgtoricmath.prepare_basis(dirname, kappa_toric)
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA


## ToricPointGeneratorMC

The ToricPointGeneratorMC works like the toric Mathematica one, but uses Monte Carlo sampling instead of Shiffman-Zelditch to generate the points. This can be much faster in the toric case, since the Veronese embedding can become pretty inefficient for toric varieties.

In [1]:
from cymetric.pointgen.pointgen_mc import ToricPointGeneratorMC

We reuse the toric date generated above, which is stored in 'toric_data.pickle' within the 'fermat_pgtoricmath' directory. This data was generated using Sage and contains the necessary GLSM structure for the toric CY hypersurface. Run the cells above in Sage if you haven't generated the data yet.

In [3]:
dirname = 'fermat_pgtoricmath'
with open(os.path.join(dirname, 'toric_data.pickle'), 'rb') as f:
    toric_data = pickle.load(f)
for key in toric_data:
    print(key)
    print(toric_data[key])

dim_cy
3
vol_j_norm
5
coeff_aK
[(-1.2107783499978841+3.0755417042654605j), (-1.7461964893542359-0.2958439132080485j), (-0.017382319548046695-1.5885147385127858j), (-1.1056651163530131-0.8594434006388215j), (-1.0773270752014996-0.44963898114534623j), (0.6342862295390801-0.6810665923093405j), (-1.463937730095888+0.877697319627733j), (-0.7122473741748493+1.759470378596264j), (0.23108940772170963+0.5912131528277726j), (-0.6908252698293487-0.6715487057800472j), (1.0262293827358568-1.9165359557710742j), (-0.6144349928998802+1.1315345315388472j), (0.8801973994604347+2.470317194746462j), (0.21613556018435437-1.3073991244171983j), (1.4581254526373584+0.5826760771808603j), (-0.35895813973212387-0.9595222194481265j), (-1.1448813789597416-0.0665174122260354j), (0.6867618147009131+0.7425453871359471j), (0.6752758348270502-0.33171461955608494j), (0.05000922129979441-0.7173088466165982j), (-0.12367348745995488+0.647799645145801j), (0.6686491786324991+0.712628149227071j), (-0.5977622437209195-1.078451

In [5]:
coefficients = np.ones(5)
monomials = 5 * np.eye(5, dtype=np.int64)
kmoduli = np.ones(1)
toric_data["coeff_aK"] = coefficients
toric_data["exp_aK"] = monomials
n_p = 100000
dirname = 'fermat_pgtoricmc'

pgtoricmc = ToricPointGeneratorMC(toric_data, kmoduli)
kappa_toric_mc = pgtoricmc.prepare_dataset(n_p, dirname)

MCpointgen:INFO:Finding initial point...
MCpointgen:INFO:Tuning step size (target acceptance 30%)...
MCpointgen:INFO:  -> step_size = 33.72
MCpointgen:INFO:Estimating thinning from autocorrelation...
MCpointgen:INFO:  -> thin = 2
MCpointgen:INFO:Running chain: burn_in = 100, thin = 2, n_p = 100000
MCMC (sampling): 100%|██████████| 200100/200100 [02:04<00:00, 1610.41step/s, accept=80.4%, pts=1e+5] 
MCpointgen:INFO:Acceptance rate: 0.804 (160849/200100)
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: divide by zero encountered in det
  r = _umath_linalg.det(a, signature=signature)
/Users/ruehle/venv-ml/lib/python3.9/site-packages/numpy/linalg/linalg.py:2180: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [6]:
pgtoricmc.prepare_basis(dirname, kappa_toric_mc)
with open(os.path.join(dirname, 'basis.pickle'), 'rb') as f:
    basis = pickle.load(f)
for key in basis:
    print(key)

DQDZB0
DQDZF0
QB0
QF0
NFOLD
AMBIENT
KMODULI
NHYPER
INTNUMS
KAPPA
